# Step 3 · Baselines

**Status:** All-positive, E5 raw and E5 standardized runs completed.

Reproduce the existing baseline evaluation from saved scores. No GPU is required by default. A new embedding run is optional and explicitly gated. Majority-class and English graded-wsd comparisons remain planned, not silently added here.

Start with a fresh runtime and execute cells in order. Read each section before running it. The original combined notebook and its outputs are retained as historical evidence.

## 1. Connect to the project
CPU is sufficient unless you explicitly enable new model inference. This uses the existing Drive project; it does not delete or recopy it.

In [ ]:
from pathlib import Path
import sys, os
try:
    from google.colab import drive
except ImportError:
    PROJECT = Path.cwd().resolve()
    if PROJECT.name == 'notebooks':
        PROJECT = PROJECT.parent
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/semeval2027/semeval2027_colab')
assert (PROJECT/'src/semeval_task3/data.py').is_file(), f'Edit PROJECT to your project root: {PROJECT}'
assert (PROJECT/'notebooks/project_utils.py').is_file(), 'Use the organized project, including notebooks/project_utils.py.'
sys.path.insert(0, str(PROJECT/'src'))
sys.path.insert(0, str(PROJECT/'notebooks'))
os.environ['PYTHONPATH'] = str(PROJECT/'src')
from project_utils import (run_checked, new_run, save_json, read_scores, save_scores,
                           frozen, standardized_by_word, record_environment, save_evaluation, digest)
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, word_bootstrap, read_submission
print('Project:', PROJECT)
print('Existing files stay in place. New outputs use unique directories under outputs/runs/.')

## 2. Choose replay or a deliberate new embedding run

In [ ]:
RUN_NEW_INFERENCE = False  # Keep False to reproduce existing results without model inference.
MODEL_ID = 'intfloat/multilingual-e5-large'
MODEL_REVISION = None  # Fresh runs resolve and record an immutable Hub commit before loading.
run_checked([sys.executable,'-m','pip','install','--quiet','numpy'],cwd=PROJECT)
import numpy as np
dev = load_dataset(PROJECT/'work/data/dev',with_gold=True)
RUN_DIR = new_run(PROJECT,'step03_baselines')
provenance = dict(mode='new_inference' if RUN_NEW_INFERENCE else 'legacy_score_replay',model=MODEL_ID)

## 3. Obtain frozen input-only scores
 Replay reads `outputs/e5_raw_scores.json`. Fresh inference follows the original configuration: `query: ` on both texts, normalized embeddings, 512-token limit, batch size 64. Original model revision was not recorded; replay preserves the actual saved scores without inventing it.

In [ ]:
import time
if not RUN_NEW_INFERENCE:
    cache = PROJECT/'outputs/e5_raw_scores.json'
    raw_scores = read_scores(cache,dev)
    provenance.update(source=str(cache.relative_to(PROJECT)),source_sha256=digest(cache),model_revision='unknown for historical run')
else:
    run_checked([sys.executable,'-m','pip','install','--quiet','sentence-transformers'],cwd=PROJECT)
    import torch
    from huggingface_hub import HfApi
    from sentence_transformers import SentenceTransformer
    assert torch.cuda.is_available(), 'Select a GPU runtime before deliberately running inference.'
    revision = MODEL_REVISION or HfApi().model_info(MODEL_ID).sha
    model = SentenceTransformer(MODEL_ID,revision=revision,device='cuda')
    model.max_seq_length = 512
    definitions = sorted({u.definition for u in dev.usages})
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); start=time.perf_counter()
    vectors = model.encode(['query: '+d for d in definitions],normalize_embeddings=True,batch_size=64)
    definition_vectors = dict(zip(definitions,vectors))
    vectors = model.encode(['query: '+u.text for u in dev.usages],normalize_embeddings=True,batch_size=64,show_progress_bar=True)
    raw_scores = {u.key:float(np.dot(definition_vectors[u.definition],v)) for u,v in zip(dev.usages,vectors)}
    torch.cuda.synchronize()
    provenance.update(model_revision=revision,seconds=time.perf_counter()-start,gpu=torch.cuda.get_device_name(),
                      peak_gpu_gib=torch.cuda.max_memory_allocated()/1024**3,batch_size=64,max_seq_length=512,prefix='query: ')
    del model
    torch.cuda.empty_cache()
save_scores(RUN_DIR/'e5_raw_scores.json',raw_scores)
record_environment(PROJECT,RUN_DIR/'manifest.json',**provenance)
print('Scores saved before evaluation:',RUN_DIR)

## 4. Evaluate with the original threshold grids
 Each variant is evaluated separately using nested leave-one-word-out. Per-word standardization uses the held-out word's **unlabeled input distribution**; report this as a transductive variant, not training-only normalization. It does not use the word's labels.

In [ ]:
from semeval_task3.evaluation import write_submission
raw_grid = tuple([-1.0]+[round(x,4) for x in np.arange(0.50,1.0001,0.0025)])
z_grid = tuple([-100.0]+[round(x,2) for x in np.arange(-3.0,3.001,0.05)])
z_scores = standardized_by_word(dev,raw_scores)
variants = [('e5_raw',raw_scores,raw_grid),('e5_z',z_scores,z_grid)]
results = {}
all_positive = score(dev.gold,{k:1 for k in dev.keys})
write_submission(RUN_DIR/'all_positive.zip',{k:1 for k in dev.keys})
save_json(RUN_DIR/'all_positive.json',dict(report=all_positive,confidence_interval=word_bootstrap(all_positive)))
print('All-positive:',all_positive['average'])
for name,scores_,thresholds in variants:
    result = nested_word_cv(dev,[Candidate(name,frozen(scores_),thresholds)])
    results[name] = result
    save_evaluation(RUN_DIR,name,result,dev,PROJECT)
    save_json(RUN_DIR/(name+'_configuration.json'),dict(thresholds=thresholds,normalization='per-word unlabeled z-score' if name=='e5_z' else 'none'))
print('Saved:',RUN_DIR)

## 5. Inspect per-word results

In [ ]:
print(f"{'word':12} {'all-positive':>13} {'E5 raw':>9} {'E5 z':>9}")
for row in all_positive['per_word']:
    values = [next(r['f1'] for r in results[n]['report']['per_word'] if (r['language'],r['word'])==(row['language'],row['word'])) for n in ('e5_raw','e5_z')]
    print(f"{row['word']:12} {row['f1']:13.4f} {values[0]:9.4f} {values[1]:9.4f}")

## Recorded historical result and next work
 All-positive **0.471630**; E5 raw **0.452178**; E5 standardized **0.412366**. These values are the original run, not outputs fabricated for this reorganized notebook. Inspect the new report if a replay differs.

 **Still pending:** properly defined training-fold majority baseline; English graded-wsd on suitable permitted English data, reported separately. Current official dev data is Swedish.

 **Next:** Step 4's Qwen experiment or Step 6's diagnostics. No system selection is final.

In [ ]:
# Majority-class baseline: learn the majority from the other nine words.
# Ties choose label 0. No held-out labels influence predictions.

gold = dev.require_gold()
groups = sorted({u.group for u in dev.usages})
predictions, folds = {}, []

for held_out in groups:
    train = [u for u in dev.usages if u.group != held_out]
    test = [u for u in dev.usages if u.group == held_out]

    positives = sum(gold[u.key] for u in train)
    majority_label = int(2 * positives > len(train))

    predictions.update({u.key: majority_label for u in test})
    folds.append({
        "held_out": list(held_out),
        "training_usages": len(train),
        "training_positives": positives,
        "majority_label": majority_label,
        "test_usages": len(test),
    })

    print(f"Held out {held_out[1]:12} → predict {majority_label}")

assert set(predictions) == dev.keys

report = score(gold, predictions)
result = {
    "predictions": predictions,
    "selections": folds,
    "report": report,
    "confidence_interval": word_bootstrap(report),
}

MAJORITY_DIR = new_run(PROJECT, "step03_majority")

save_evaluation(
    MAJORITY_DIR, "majority_class", result, dev, PROJECT
)
record_environment(
    PROJECT,
    MAJORITY_DIR / "manifest.json",
    purpose="Leave-one-word-out majority-class baseline",
    rule="Pooled training-usage majority; ties predict 0",
)

print("\nSaved:", MAJORITY_DIR)

In [ ]:
# Install the official scorer's dependencies.
run_checked(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "scikit-learn", "scipy", "bcubed==1.5"],
    cwd=PROJECT,
)

from semeval_task3.evaluation import official_score
from semeval_task3.__main__ import check_agreement

baseline_dir = (
    PROJECT / "outputs/runs/step03_baselines"
    / "20261103T163804Z_b602cbfb"
)
majority_dir = (
    PROJECT / "outputs/runs/step03_majority"
    / "20261103T164425Z_91ad42f5"
)

archives = {
    "all_positive": baseline_dir / "all_positive.zip",
    "e5_raw": baseline_dir / "e5_raw.zip",
    "e5_z": baseline_dir / "e5_z.zip",
    "majority_class": majority_dir / "majority_class.zip",
}

checks = {}
for name, path in archives.items():
    local = score(dev.require_gold(), read_submission(path, dev.keys))
    official = official_score(
        path, PROJECT / "work/data/dev", PROJECT / "work/official-scorer"
    )
    check_agreement(local, official)
    checks[name] = {"local": local, "official": official}
    print(f"PASS — {name}: F1={local['average']:.6f}")

CHECK_DIR = new_run(PROJECT, "step03_official_checks")
save_json(CHECK_DIR / "official_agreement.json", checks)
record_environment(
    PROJECT, CHECK_DIR / "manifest.json",
    purpose="Official scorer agreement for all four baselines",
)
print("Saved:", CHECK_DIR)

In [ ]:
import csv

table_path = CHECK_DIR / "baseline_results.csv"

rows = [
    {
        "system": name,
        "mean_per_word_f1": result["local"]["average"],
        "official_scorer_agreement": True,
        "prediction_zip": str(archives[name]),
    }
    for name, result in checks.items()
]
rows.sort(key=lambda row: row["mean_per_word_f1"], reverse=True)

with table_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print("Saved:", table_path)

In [ ]:
import subprocess
from collections import Counter

WIC_REPO = "https://github.com/semantic-web-company/wic-tsv.git"
WIC_DIR = PROJECT / "work/external/wic-tsv"
WIC_DIR.parent.mkdir(parents=True, exist_ok=True)

if not WIC_DIR.exists():
    run_checked(
        ["git", "clone", "--depth", "1", WIC_REPO, str(WIC_DIR)],
        cwd=PROJECT,
    )

commit = subprocess.run(
    ["git", "-C", str(WIC_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

audit = {
    "source": WIC_REPO,
    "commit": commit,
    "dataset_license": "CC-BY-4.0",
    "purpose": "External English evaluation; separate from SemEval",
    "files": {},
    "splits": {},
}

for folder, prefix in [("Training", "train"), ("Development", "dev")]:
    contents = {}
    for kind in ["examples", "definitions", "labels"]:
        path = WIC_DIR / "data/en" / folder / f"{prefix}_{kind}.txt"
        assert path.is_file(), f"Missing: {path}"
        contents[kind] = path.read_text(encoding="utf-8").splitlines()
        audit["files"][str(path.relative_to(WIC_DIR))] = digest(path)

    sizes = {kind: len(lines) for kind, lines in contents.items()}
    assert len(set(sizes.values())) == 1, f"Misaligned files: {sizes}"
    assert sizes["examples"] > 0, "Empty split"

    labels = [x.strip() for x in contents["labels"]]
    assert set(labels) <= {"T", "F"}, f"Unexpected labels: {set(labels)}"

    audit["splits"][prefix] = {
        "examples": len(labels),
        "label_counts": dict(Counter(labels)),
    }

    print(f"\n{folder}: {len(labels)} examples; labels={dict(Counter(labels))}")
    print("First example:", contents["examples"][0])
    print("Definition:", contents["definitions"][0])
    print("Label:", labels[0])

WIC_AUDIT_DIR = new_run(PROJECT, "step03_english_data")
save_json(WIC_AUDIT_DIR / "data_audit.json", audit)
print("\nDataset commit:", commit)
print("Audit saved:", WIC_AUDIT_DIR)

In [ ]:
import re

def load_wic_split(folder, prefix):
    base = WIC_DIR / "data/en" / folder

    def lines(kind):
        return (base / f"{prefix}_{kind}.txt").read_text(
            encoding="utf-8"
        ).splitlines()

    examples, definitions, labels = (
        lines("examples"), lines("definitions"), lines("labels")
    )
    assert len(examples) == len(definitions) == len(labels)

    rows = []
    for i, (example, definition, label) in enumerate(
        zip(examples, definitions, labels)
    ):
        word, index, text = example.split("\t", 2)
        index = int(index)
        tokens = list(re.finditer(r"\S+", text))
        assert 0 <= index < len(tokens), f"{prefix}:{i}: invalid target index"
        assert definition.strip(), f"{prefix}:{i}: empty definition"
        assert label.strip() in {"T", "F"}

        start, end = tokens[index].span()
        rows.append({
            "id": f"wic_tsv_en_{prefix}_{i:05d}",
            "word": word,
            "text": text,
            "start": start,
            "end": end,
            "marked_text": text[:start] + "<t>" + text[start:end]
                           + "</t>" + text[end:],
            "definition": definition,
            "label": int(label.strip() == "T"),
        })
    return rows

wic_train = load_wic_split("Training", "train")
wic_dev = load_wic_split("Development", "dev")

def pair_key(row):
    return (
        " ".join(row["marked_text"].casefold().split()),
        " ".join(row["definition"].casefold().split()),
    )

shared_pairs = set(map(pair_key, wic_train)) & set(map(pair_key, wic_dev))
shared_words = (
    {r["word"].casefold() for r in wic_train}
    & {r["word"].casefold() for r in wic_dev}
)

save_json(WIC_AUDIT_DIR / "prepared_train.json", wic_train)
save_json(WIC_AUDIT_DIR / "prepared_dev.json", wic_dev)
save_json(WIC_AUDIT_DIR / "overlap_check.json", {
    "shared_usage_definition_pairs": len(shared_pairs),
    "shared_words": sorted(shared_words),
})

print("Prepared:", len(wic_train), "train /", len(wic_dev), "dev")
print("Shared usage–definition pairs:", len(shared_pairs))
print("Shared words:", len(shared_words))
for row in [wic_train[0], wic_dev[0]]:
    print("\nMarked context:", row["marked_text"])
    print("Definition:", row["definition"])
print("\nSaved:", WIC_AUDIT_DIR)

In [ ]:
run_checked(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "transformers", "huggingface_hub", "safetensors"],
    cwd=PROJECT,
)

import torch
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForSequenceClassification

assert torch.cuda.is_available(), "Connect a GPU runtime before continuing."

GWSD_MODEL_ID = "ChangeIsKey/graded-wsd"
GWSD_REVISION = HfApi().model_info(GWSD_MODEL_ID).sha
GWSD_RUN = new_run(PROJECT, "step03_graded_wsd_english")

gwsd_tokenizer = AutoTokenizer.from_pretrained(
    GWSD_MODEL_ID, revision=GWSD_REVISION
)
gwsd_model = AutoModelForSequenceClassification.from_pretrained(
    GWSD_MODEL_ID,
    revision=GWSD_REVISION,
    use_safetensors=True,
).to("cuda").eval()

assert gwsd_model.config.num_labels == 1, "Expected one regression output."

# Verify the checkpoint recognizes the target markers.
for marker in ["<t>", "</t>"]:
    ids = gwsd_tokenizer.encode(marker, add_special_tokens=False)
    assert len(ids) == 1 and ids[0] != gwsd_tokenizer.unk_token_id, (
        f"Unexpected tokenization for {marker}: {ids}"
    )

# Use training examples only for this initial check.
smoke_rows = [wic_train[0], next(r for r in wic_train if r["label"] == 0)]
smoke_inputs = [
    f'{r["marked_text"]} </s></s> {r["definition"]}'
    for r in smoke_rows
]
batch = gwsd_tokenizer(
    smoke_inputs, padding=True, truncation=False, return_tensors="pt"
).to("cuda")
assert batch["input_ids"].shape[1] <= 512, "Example exceeds input limit."

with torch.inference_mode():
    smoke_scores = gwsd_model(**batch).logits.squeeze(-1).float().cpu()

assert torch.isfinite(smoke_scores).all(), "Non-finite model output."

for row, value in zip(smoke_rows, smoke_scores.tolist()):
    print(f"\nLabel={row['label']} | graded score={value:.4f}")
    print(row["marked_text"])
    print("Definition:", row["definition"])

save_json(GWSD_RUN / "smoke_test.json", [
    {"id": row["id"], "label": row["label"], "score": value}
    for row, value in zip(smoke_rows, smoke_scores.tolist())
])
record_environment(
    PROJECT, GWSD_RUN / "manifest.json",
    model_id=GWSD_MODEL_ID,
    model_revision=GWSD_REVISION,
    dataset_commit=commit,
    gpu=torch.cuda.get_device_name(0),
    purpose="External English WiC-TSV evaluation; frozen model weights",
)

print("\nGPU:", torch.cuda.get_device_name(0))
print("Model revision:", GWSD_REVISION)
print("Saved:", GWSD_RUN)

In [ ]:
import time
from tqdm.auto import tqdm

GWSD_BATCH_SIZE = 16

def score_wic_split(rows, split_name):
    texts = [
        f'{r["marked_text"]} </s></s> {r["definition"]}'
        for r in rows
    ]

    # Check lengths before inference so nothing is silently truncated.
    encoded = gwsd_tokenizer(
        texts, padding=False, truncation=False
    )
    lengths = [len(ids) for ids in encoded["input_ids"]]
    too_long = [rows[i]["id"] for i, n in enumerate(lengths) if n > 512]
    assert not too_long, (
        f"{len(too_long)} examples exceed 512 tokens. "
        f"Stop and report these IDs: {too_long[:10]}"
    )

    scores = []
    torch.cuda.synchronize()
    started = time.perf_counter()

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(rows), GWSD_BATCH_SIZE),
            desc=f"Scoring {split_name}",
        ):
            stop = min(start + GWSD_BATCH_SIZE, len(rows))
            features = [
                {key: values[i] for key, values in encoded.items()}
                for i in range(start, stop)
            ]
            batch = gwsd_tokenizer.pad(
                features, padding=True, return_tensors="pt"
            ).to("cuda")

            values = gwsd_model(**batch).logits.squeeze(-1).float().cpu()
            assert torch.isfinite(values).all(), "Non-finite scores"
            scores.extend(values.tolist())

    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    assert len(scores) == len(rows)

    output = GWSD_RUN / f"{split_name}_scores.json"
    save_json(output, {
        "model_revision": GWSD_REVISION,
        "seconds": seconds,
        "batch_size": GWSD_BATCH_SIZE,
        "max_input_tokens": max(lengths),
        "predictions": [
            {"id": row["id"], "score": value}
            for row, value in zip(rows, scores)
        ],
    })
    print(
        f"{split_name}: saved {len(scores)} scores; "
        f"{seconds:.1f}s; longest input={max(lengths)} tokens"
    )
    return scores

gwsd_train_scores = score_wic_split(wic_train, "train")
gwsd_dev_scores = score_wic_split(wic_dev, "dev")
print("\nSaved:", GWSD_RUN)

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, precision_recall_fscore_support

train_y = np.array([r["label"] for r in wic_train], dtype=int)
train_s = np.array(gwsd_train_scores, dtype=float)
assert len(train_y) == len(train_s) and np.isfinite(train_s).all()

# Select threshold on training data only.
p, r, thresholds = precision_recall_curve(train_y, train_s)
train_f1 = np.divide(
    2 * p[:-1] * r[:-1],
    p[:-1] + r[:-1],
    out=np.zeros_like(p[:-1]),
    where=(p[:-1] + r[:-1]) > 0,
)
best = int(np.argmax(train_f1))  # Ties: lowest threshold.
gwsd_threshold = float(thresholds[best])

threshold_record = {
    "threshold": gwsd_threshold,
    "rule": "score >= threshold",
    "selected_on": "WiC-TSV English training split only",
    "objective": "Pooled positive-class binary F1",
    "training_f1": float(train_f1[best]),
    "tie_break": "Lowest threshold",
}
save_json(GWSD_RUN / "frozen_threshold.json", threshold_record)

# Evaluate development data only after freezing the threshold.
dev_y = np.array([r["label"] for r in wic_dev], dtype=int)
dev_s = np.array(gwsd_dev_scores, dtype=float)
assert len(dev_y) == len(dev_s) and np.isfinite(dev_s).all()

dev_predictions = (dev_s >= gwsd_threshold).astype(int)
all_positive_predictions = np.ones_like(dev_y)

def binary_metrics(predictions):
    precision, recall, f1, _ = precision_recall_fscore_support(
        dev_y, predictions, average="binary", zero_division=0
    )
    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "predicted_positives": int(predictions.sum()),
    }

# Additional per-word diagnostic using our shared evaluator.
en_gold = {
    ("EN", row["word"], row["id"]): row["label"]
    for row in wic_dev
}
def per_word_report(predictions):
    return score(en_gold, {
        ("EN", row["word"], row["id"]): int(prediction)
        for row, prediction in zip(wic_dev, predictions)
    })

model_report = per_word_report(dev_predictions)
positive_report = per_word_report(all_positive_predictions)

english_results = {
    "dataset": "WiC-TSV English development — external evaluation",
    "n": len(wic_dev),
    "threshold_selection": threshold_record,
    "graded_wsd": binary_metrics(dev_predictions),
    "all_positive": binary_metrics(all_positive_predictions),
    "per_word_diagnostic": {
        "graded_wsd": model_report,
        "all_positive": positive_report,
        "graded_wsd_ci": word_bootstrap(model_report),
        "paired_difference_ci": word_bootstrap(model_report, other=positive_report),
    },
    "limitations": [
        "98 words overlap between training and development.",
        "Not a historical-text or official SemEval evaluation.",
        "Overlap with pretrained model training data is unknown.",
    ],
}
save_json(GWSD_RUN / "evaluation.json", english_results)
save_json(GWSD_RUN / "dev_predictions.json", [
    {"id": row["id"], "prediction": int(prediction)}
    for row, prediction in zip(wic_dev, dev_predictions)
])

print(f"Frozen threshold: {gwsd_threshold:.6f}")
for name in ["graded_wsd", "all_positive"]:
    m = english_results[name]
    print(
        f"{name}: F1={m['f1']:.4f}, "
        f"precision={m['precision']:.4f}, recall={m['recall']:.4f}"
    )
print(f"Graded-WSD mean per-word F1: {model_report['average']:.4f}")
print("Per-word bootstrap:", english_results["per_word_diagnostic"]["graded_wsd_ci"])
print("Saved:", GWSD_RUN / "evaluation.json")

In [ ]:
import csv
import json

# Read saved results so the summary reflects persisted artifacts.
with (CHECK_DIR / "baseline_results.csv").open(
    encoding="utf-8", newline=""
) as f:
    swedish_rows = list(csv.DictReader(f))

english = json.loads(
    (GWSD_RUN / "evaluation.json").read_text(encoding="utf-8")
)
model_manifest = json.loads(
    (GWSD_RUN / "manifest.json").read_text(encoding="utf-8")
)

SUMMARY_DIR = new_run(PROJECT, "step03_summary")

summary = {
    "step": 3,
    "status": "complete",
    "swedish_baselines": swedish_rows,
    "english_external_evaluation": english,
    "english_model_manifest": model_manifest,
    "artifacts": {
        "swedish_official_checks": str(CHECK_DIR),
        "english_data_audit": str(WIC_AUDIT_DIR),
        "english_model_run": str(GWSD_RUN),
    },
}
save_json(SUMMARY_DIR / "summary.json", summary)

sv_table = "\n".join(
    f"| {row['system']} | {float(row['mean_per_word_f1']):.6f} |"
    for row in swedish_rows
)
en_table = "\n".join(
    f"| {name} | {english[name]['f1']:.4f} | "
    f"{english[name]['precision']:.4f} | {english[name]['recall']:.4f} |"
    for name in ["graded_wsd", "all_positive"]
)

report = f"""# Step 3 — Baselines completed

## Official Swedish development data
All four baselines passed agreement checks against the official scorer.
Scores below are development results, not official test leaderboard scores.

| System | Mean per-word F1 |
|---|---:|
{sv_table}

All-positive remains the strongest of these four baselines.
The E5 comparisons did not establish an improvement over all-positive.

## External English comparison: WiC-TSV
Frozen graded-WSD weights; threshold selected using training labels only.
This is a calibrated external baseline, not a label-free evaluation.
Threshold: {english['threshold_selection']['threshold']:.6f}
Development examples: {english['n']}

| System | Pooled binary F1 | Precision | Recall |
|---|---:|---:|---:|
{en_table}

English results are separate from the Swedish SemEval results.
The additional per-word diagnostic is stored in evaluation.json.

## Limitations
- English training and development share 98 words.
- No shared normalized usage–definition pairs were found.
- Overlap with the model's pretraining/fine-tuning data is unknown.
- WiC-TSV does not establish historical or multilingual performance.
- English per-word groups are mostly single examples.

## Sources
- WiC-TSV: https://aclanthology.org/2021.eacl-main.140/
- Dataset: https://github.com/semantic-web-company/wic-tsv
- Model: https://huggingface.co/ChangeIsKey/graded-wsd

## Saved artifacts
- Swedish scorer checks: {CHECK_DIR}
- English data audit: {WIC_AUDIT_DIR}
- English model, scores and evaluation: {GWSD_RUN}
- Consolidated summary: {SUMMARY_DIR}

## Next
Step 4: review the existing Qwen candidate and plan the remaining candidates.
"""

(SUMMARY_DIR / "README.md").write_text(report, encoding="utf-8")
(PROJECT / "docs").mkdir(exist_ok=True)
(PROJECT / "docs/STEP03_RESULTS.md").write_text(report, encoding="utf-8")

print("STEP 3 COMPLETE")
print("Summary:", SUMMARY_DIR)
print("Project report:", PROJECT / "docs/STEP03_RESULTS.md")